In [30]:
import pandas as pd
import numpy as np

In [31]:
from sklearn.datasets import load_breast_cancer
breast_cancer = load_breast_cancer()
X, y = breast_cancer.data, breast_cancer.target

### Собственная реализация бэггинга

#### Будем использовать собственную реализация логистической регрессии в качестве базовой модели

In [32]:
from sklearn.base import BaseEstimator, ClassifierMixin
class MyLogisticRegression(BaseEstimator, ClassifierMixin):
    
    def __init__(self, fit_intercept = True):
        self.fit_intercept = fit_intercept
        self.threshold = 0.5
        
    @staticmethod
    def default_log_loss(y, p):
        # обрежем значения, чтобы предотвратить log(0)
        eps = 1e-15
        p = np.clip(p, eps, 1 - eps)
        return -1 * np.mean(y * np.log(p) + (1 - y)* np.log(1 - p))
        
    def fit(self, X, y, learning_rate = 1e-4, tol = 1e-7, max_iter = 10000, mu = 0.9, loss_func = default_log_loss):
        if loss_func is None:
            loss_func = default_log_loss
        
        n_objects, self.n_features = X.shape
        self.coef = np.zeros(self.n_features)
        self.intercept = 0
        
        def loss_difference(curr_loss, previous_loss):
            return abs(curr_loss - previous_loss) / max(1, previous_loss)
        self.loss = loss_func(y, self.predict_proba(X))
        self.last_loss = 1e6
        
        v_coef = np.zeros(self.n_features)
        v_intercept = 0
        
        while loss_difference(self.loss, self.last_loss) > tol and max_iter>0:
            coef_lookahead = self.coef + mu * v_coef
            intercept_lookahead = self.intercept + mu * v_intercept

            predictions = self.predict_proba(X, coef_lookahead, intercept_lookahead)
            dw = (1.0 / n_objects) * (X.T @ (predictions - y))
            db = (1.0 / n_objects) * np.sum(predictions - y)

            v_coef = mu * v_coef - learning_rate * dw
            v_intercept = mu * v_intercept - learning_rate * db

            self.coef += v_coef
            self.intercept += v_intercept

            self.last_loss = self.loss
            predictions_proba = self.predict_proba(X)
            self.loss = loss_func(y, predictions_proba)
            max_iter -= 1
        return self

    def predict_proba(self, X, coef = None, intercept = None):
        def sigmoid(logit):
            return 1.0 / (1.0 + np.exp(-logit))
        if coef is None:
            coef = self.coef
        if intercept is None:
            intercept = self.intercept
        logit = X @ coef + intercept
        sigm_output = sigmoid(logit)
        return sigm_output
        
    def predict(self, X):
        return (self.predict_proba(X) >= self.threshold).astype(int)

#### Построение ансамбля

In [33]:
from sklearn.base import BaseEstimator, RegressorMixin
from joblib import Parallel, delayed

class My_BaggingClassifier(BaseEstimator, RegressorMixin):
    def __init__(self, n_estimators = 5, bootstrap = True, max_samples = 0.8, max_features = 0.8, fit_intercept = True, random_state = 76, n_jobs = -1):
        self.n_estimators = n_estimators 
        self.bootstrap = bootstrap
        self.max_samples = max_samples 
        self.max_features = max_features 
        self.fit_intercept = fit_intercept
        self.random_state = random_state
        self.rng = np.random.RandomState(random_state)
        self.estimators_ = []
        self.feature_indices_ = []
        self.n_jobs = n_jobs
    
        
    def bootstrap_samples(self, X, y):
        n_rows, n_cols = X.shape
        
        bootstrap_size = int(n_rows * self.max_samples)
        bootstrap_cols = int(n_cols * self.max_features)
        
        samples = []
        
        for _ in range(self.n_estimators):
            row_indices = self.rng.choice(
                n_rows, 
                size = bootstrap_size, 
                replace = True
            )

            cols_indices = self.rng.choice(
                n_cols,
                size = bootstrap_cols,
                replace = False
            )

            self.feature_indices_.append(cols_indices)
            
            
            X_sample = X[row_indices][:, cols_indices]
            y_sample = y[row_indices]
            
            samples.append((X_sample, y_sample, cols_indices))
            
        return samples

    def _fit_estimator(self, X_sample, y_sample, col_indices):
        model = MyLogisticRegression(fit_intercept = self.fit_intercept)
        model.fit(X_sample, y_sample)
        return model, col_indices
    
    def fit(self, X, y):
        self.n_samples = X.shape[0]
        self.classes_ = np.unique(y)
        cols = np.arange(X.shape[1])
        base_models_data = [(X, y, cols) for _ in range(self.n_estimators)]
        if self.bootstrap:
            base_models_data = self.bootstrap_samples(X, y)
        else:
            self.feature_indices_ = [np.arange(X.shape[1]) for _ in range(self.n_estimators)]

        results = Parallel(n_jobs=self.n_jobs)(
            delayed(self._fit_estimator)(X_s, y_s, cols)
            for (X_s, y_s, cols) in base_models_data
        )

        self.estimators_ = [res[0] for res in results]
        self.feature_indices_ = [res[1] for res in results]

        return self
        
    def predict(self, X):
        n_samples = X.shape[0]
        n_classes = len(self.classes_)
        predictions = np.zeros((len(self.estimators_), n_samples), dtype = int)
        
        for i, (model, feature_indices) in enumerate(zip(self.estimators_, self.feature_indices_)):
            X_sub = X[:, feature_indices]
            predictions[i, :] = model.predict(X_sub)

        one_hot = np.eye(n_classes)[predictions]
        vote_sum = one_hot.sum(axis = 0)
        return self.classes_[np.argmax(vote_sum, axis = 1)]

## Сравнение по качеству

In [34]:
from sklearn.model_selection import cross_val_score

### Моя реализация

In [35]:
%%time
my_bagging_classifier = My_BaggingClassifier(n_estimators = 5, bootstrap = True)
print(f'Accuracy: {np.mean(cross_val_score(my_bagging_classifier, X, y, cv = 7, scoring = 'accuracy'))}')

Accuracy: 0.9262915645029466
CPU times: total: 625 ms
Wall time: 18.5 s


### Готовая реализация sklearn

In [36]:
%%time
from sklearn.ensemble import BaggingClassifier

bagging_classifier = BaggingClassifier(
    n_estimators = 5, 
    bootstrap = True, 
    max_samples = 0.8,
    max_features = 0.8,
    random_state = 42
)

print(f'Accuracy: {np.mean(cross_val_score(bagging_classifier, X, y, cv = 7, scoring = 'accuracy'))}')

Accuracy: 0.9490471888845872
CPU times: total: 422 ms
Wall time: 435 ms
